# CatBoost - Treinamento e Avaliação

In [2]:
# Instalação dos pacotes necessários
import sys
!{sys.executable} -m pip install catboost scikit-learn pandas numpy optuna

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.


In [3]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold

from catboost import CatBoostClassifier
import optuna
from optuna.samplers import TPESampler

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42

/Users/gabrielvanderlei/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Funções Auxiliares

In [4]:
def ensure_dirs():
    os.makedirs("trained_models_catboost", exist_ok=True)
    os.makedirs("results", exist_ok=True)
    os.makedirs("optuna_results", exist_ok=True)


def load_best_params(dataset_name: str):
    """Carrega hiperparâmetros do Optuna."""
    path = f"optuna_results/{dataset_name}_catboost_best.json"
    with open(path, "r") as f:
        data = json.load(f)
    return data["best_params"], data.get("tune_time", np.nan)


def load_dataset(dataset_name: str):
    """Carrega os datasets já separados."""
    train = pd.read_csv(f"train_datasets/{dataset_name}.csv")
    test = pd.read_csv(f"test_datasets/{dataset_name}.csv")
    return train, test

In [5]:
def preprocess(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """Preprocessamento padronizado para treino e teste."""

    # Fit LabelEncoder no treino
    le_target = LabelEncoder()
    y_train = le_target.fit_transform(train_df["current_target_class"].values)
    y_test = le_target.transform(test_df["current_target_class"].values)

    cat_cols = train_df.select_dtypes(include="object").columns.tolist()
    num_cols = (
        train_df.select_dtypes(include=np.number)
        .drop(columns=["current_target_class"])
        .columns.tolist()
    )

    train_copy = train_df.copy()
    test_copy = test_df.copy()

    # Codificar categóricas
    cat_indices = []
    cat_encoders = {}

    if len(cat_cols) > 0:
        for c in cat_cols:
            enc = LabelEncoder()
            train_copy[c] = enc.fit_transform(train_copy[c].astype(str))

            # Handle unseen categories in test
            test_vals = test_copy[c].astype(str)
            test_copy[c] = test_vals.apply(
                lambda x: enc.transform([x])[0] if x in enc.classes_ else -1
            )
            cat_encoders[c] = enc

        cat_indices = list(range(len(num_cols), len(num_cols) + len(cat_cols)))

    # Features numéricas
    X_train_num = train_copy[num_cols].values.astype(np.float32)
    X_test_num = test_copy[num_cols].values.astype(np.float32)

    # Normalização
    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train_num)
    X_test_num = scaler.transform(X_test_num)

    # Concatenar
    if len(cat_cols) > 0:
        X_train_cat = train_copy[cat_cols].values.astype(np.int32)
        X_test_cat = test_copy[cat_cols].values.astype(np.int32)
        X_train = np.hstack([X_train_num, X_train_cat])
        X_test = np.hstack([X_test_num, X_test_cat])
    else:
        X_train = X_train_num
        X_test = X_test_num

    return X_train, X_test, y_train, y_test, cat_indices, le_target

## Métricas de Avaliação

In [6]:
def gmean_score(y_true, y_pred, eps=1e-9):
    """G-Mean: média geométrica dos recalls por classe."""
    classes = np.unique(y_true)
    recalls = []

    for c in classes:
        tp = np.sum((y_true == c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))
        recalls.append(tp / (tp + fn + eps))

    return float(np.prod(recalls) ** (1.0 / len(recalls)))


def compute_auc_ovo(y_true, probs):
    """Cálculo robusto do AUC OVO."""
    y_true = np.asarray(y_true)
    probs = np.asarray(probs)
    unique_classes = np.unique(y_true)

    if len(unique_classes) < 2:
        return np.nan

    try:
        if len(unique_classes) > 2:
            return roc_auc_score(
                y_true,
                probs,
                multi_class="ovo",
                average="macro",
                labels=unique_classes
            )
        else:
            return roc_auc_score(y_true, probs[:, 1])
    except Exception:
        return np.nan


def evaluate_split(model, X, y):
    """Avaliação completa em um split (train/test)."""
    start = time.time()
    probs = model.predict_proba(X)
    tempo_predict = time.time() - start

    probs = np.asarray(probs)
    y = np.asarray(y)

    preds = np.argmax(probs, axis=1)

    return {
        "auc_ovo": compute_auc_ovo(y, probs),
        "mean_acc": accuracy_score(y, preds),
        "g_mean": gmean_score(y, preds),
        "mean_cross_entropy": log_loss(y, probs),
        "tempo_predict": tempo_predict,
    }

## Função de Treinamento

In [7]:
def train_final_model(dataset_name: str):
    """Treina modelo final com melhores hiperparâmetros."""
    print(f"\n{'='*60}")
    print(f"Treinando CatBoost para: {dataset_name}")
    print(f"{'='*60}")

    # Carregar melhores parâmetros
    best_params, tempo_tune = load_best_params(dataset_name)
    print(f"Melhores hiperparâmetros: {best_params}")

    # Carregar dados
    train_df, test_df = load_dataset(dataset_name)

    # Preprocessar
    X_train, X_test, y_train, y_test, cat_indices, le_target = preprocess(
        train_df, test_df
    )

    print(f"  Train: {X_train.shape[0]} amostras")
    print(f"  Test: {X_test.shape[0]} amostras")
    print(f"  Features: {X_train.shape[1]}")
    print(f"  Classes: {len(np.unique(y_train))}")

    # Configurar modelo
    model_params = {
        **best_params,
        "random_seed": SEED,
        "verbose": False,
        "thread_count": -1,
        "task_type": "CPU",
    }

    model = CatBoostClassifier(**model_params)

    # Treinar
    start_train = time.time()

    if len(cat_indices) > 0:
        model.fit(X_train, y_train, cat_features=cat_indices, verbose=False)
    else:
        model.fit(X_train, y_train, verbose=False)

    tempo_train = time.time() - start_train
    print(f"  Tempo de treino: {tempo_train:.2f}s")

    # Salvar modelo
    model_path = f"trained_models_catboost/{dataset_name}_catboost.cbm"
    model.save_model(model_path)
    print(f"  Modelo salvo: {model_path}")

    # Avaliar
    train_metrics = evaluate_split(model, X_train, y_train)
    test_metrics = evaluate_split(model, X_test, y_test)

    print(f"\n  Resultados no TREINO:")
    print(f"    AUC OVO: {train_metrics['auc_ovo']:.4f}")
    print(f"    Accuracy: {train_metrics['mean_acc']:.4f}")
    print(f"    G-Mean: {train_metrics['g_mean']:.4f}")
    print(f"    Cross-Entropy: {train_metrics['mean_cross_entropy']:.4f}")

    print(f"\n  Resultados no TESTE:")
    print(f"    AUC OVO: {test_metrics['auc_ovo']:.4f}")
    print(f"    Accuracy: {test_metrics['mean_acc']:.4f}")
    print(f"    G-Mean: {test_metrics['g_mean']:.4f}")
    print(f"    Cross-Entropy: {test_metrics['mean_cross_entropy']:.4f}")

    # Construir saída
    rows = []

    for split, metrics in [("train", train_metrics), ("test", test_metrics)]:
        rows.append({
            "split": split,
            "nome_modelo": "CatBoost",
            "dataset": dataset_name,
            "tempo_tune": tempo_tune,
            "tempo_train": tempo_train,
            "auc_ovo": metrics["auc_ovo"],
            "mean_acc": metrics["mean_acc"],
            "g_mean": metrics["g_mean"],
            "mean_cross_entropy": metrics["mean_cross_entropy"],
            "tempo_predict": metrics["tempo_predict"],
        })

    return rows

## Otimização de Hiperparâmetros (Tuning)

In [8]:
def tune_catboost_optuna(dataset_name: str, n_trials: int = 50):
    """
    Otimiza hiperparâmetros do CatBoost usando Optuna com validação cruzada.
    """
    print(f"\n{'='*60}")
    print(f"Tuning CatBoost para: {dataset_name}")
    print(f"{'='*60}")
    
    # Carregar dados
    train_df, test_df = load_dataset(dataset_name)
    
    # Preprocessar
    X_train, X_test, y_train, y_test, cat_indices, le_target = preprocess(
        train_df, test_df
    )
    
    print(f"  Train: {X_train.shape[0]} amostras")
    print(f"  Features: {X_train.shape[1]}")
    print(f"  Classes: {len(np.unique(y_train))}")
    print(f"  Trials: {n_trials}")
    
    # Função objetivo para o Optuna
    def objective(trial):
        params = {
            "iterations": trial.suggest_int("iterations", 50, 500),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "depth": trial.suggest_int("depth", 3, 10),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "random_seed": SEED,
            "verbose": False,
            "thread_count": -1,
            "task_type": "CPU",
        }
        
        # Validação cruzada estratificada
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        cv_scores = []
        
        for train_idx, val_idx in skf.split(X_train, y_train):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            
            model = CatBoostClassifier(**params)
            
            if len(cat_indices) > 0:
                model.fit(X_tr, y_tr, cat_features=cat_indices, verbose=False)
            else:
                model.fit(X_tr, y_tr, verbose=False)
            
            probs = model.predict_proba(X_val)
            auc = compute_auc_ovo(y_val, probs)
            
            if not np.isnan(auc):
                cv_scores.append(auc)
        
        return np.mean(cv_scores) if len(cv_scores) > 0 else 0.0
    
    # Executar otimização
    start_time = time.time()
    
    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=SEED)
    )
    
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    tune_time = time.time() - start_time
    
    # Melhores parâmetros
    best_params = study.best_params
    best_score = study.best_value
    
    print(f"\n  Melhor AUC (CV): {best_score:.4f}")
    print(f"  Tempo de tuning: {tune_time:.2f}s")
    print(f"  Melhores parâmetros: {best_params}")
    
    # Salvar resultados
    output = {
        "dataset": dataset_name,
        "best_params": best_params,
        "best_score": float(best_score),
        "tune_time": float(tune_time),
        "n_trials": n_trials,
    }
    
    output_path = f"optuna_results/{dataset_name}_catboost_best.json"
    with open(output_path, "w") as f:
        json.dump(output, f, indent=2)
    
    print(f"  Resultados salvos: {output_path}")
    
    return best_params, best_score, tune_time

In [9]:
# EXECUTAR TUNING PARA TODOS OS DATASETS
# Execute esta célula apenas se quiser rodar o tuning manualmente
# Os resultados são salvos em optuna_results/ e reutilizados automaticamente

ensure_dirs()

# Lista completa dos 30 datasets
dataset_list_tune = [
    # Datasets balanceados
    'credit-approval', 'dresses-sales', 'mfeat-morphological', 'vehicle',
    'banknote-authentication', 'analcatdata_dmft', 'MiceProtein', 'cylinder-bands',
    'semeion', 'cnae-9', 'vowel',
    # Datasets desbalanceados
    'breast-w', 'eucalyptus', 'wdbc', 'pc4', 'credit-g', 'cmc',
    'blood-transfusion-service-center', 'pc3', 'car', 'kc2',
    'steel-plates-fault', 'balance-scale', 'pc1', 'tic-tac-toe',
    'analcatdata_authorship', 'climate-model-simulation-crashes',
    'qsar-biodeg', 'diabetes', 'ilpd',
]

# Executar tuning para cada dataset
tune_results = []

for ds in dataset_list_tune:
    try:
        # Verificar se já foi tunado
        param_path = f"optuna_results/{ds}_catboost_best.json"
        
        if os.path.exists(param_path):
            print(f"\n[INFO] {ds} já foi tunado. Pulando...")
            # Carregar resultados existentes
            with open(param_path, 'r') as f:
                data = json.load(f)
                tune_results.append({
                    'dataset': ds,
                    'best_score': data['best_score'],
                    'tune_time': data['tune_time']
                })
        else:
            print(f"\n[INFO] Tunando {ds}...")
            best_params, best_score, tune_time = tune_catboost_optuna(ds, n_trials=50)
            tune_results.append({
                'dataset': ds,
                'best_score': best_score,
                'tune_time': tune_time
            })
    except Exception as e:
        print(f"\n[ERRO] Dataset {ds}: {e}")
        continue

# Resumo dos resultados de tuning
if len(tune_results) > 0:
    tune_df = pd.DataFrame(tune_results)
    print(f"\n{'='*60}")
    print("RESUMO DO TUNING")
    print(f"{'='*60}")
    print(f"Total de datasets: {len(tune_df)}")
    print(f"Tempo total de tuning: {tune_df['tune_time'].sum():.2f}s")
    print(f"AUC médio (CV): {tune_df['best_score'].mean():.4f}")
    display(tune_df.sort_values('best_score', ascending=False))


[INFO] credit-approval já foi tunado. Pulando...

[INFO] dresses-sales já foi tunado. Pulando...

[INFO] mfeat-morphological já foi tunado. Pulando...

[INFO] vehicle já foi tunado. Pulando...

[INFO] banknote-authentication já foi tunado. Pulando...

[INFO] analcatdata_dmft já foi tunado. Pulando...

[INFO] MiceProtein já foi tunado. Pulando...

[INFO] cylinder-bands já foi tunado. Pulando...

[INFO] Tunando semeion...

Tuning CatBoost para: semeion
  Train: 1003 amostras
  Features: 256
  Classes: 10
  Trials: 50


Best trial: 32. Best value: 0.997293: 100%|██████████| 50/50 [59:12<00:00, 71.05s/it]  



  Melhor AUC (CV): 0.9973
  Tempo de tuning: 3552.30s
  Melhores parâmetros: {'iterations': 433, 'learning_rate': 0.09109634495981846, 'depth': 4, 'l2_leaf_reg': 0.06283219021952564, 'random_strength': 0.006190558077203713, 'bagging_temperature': 0.03198201865774064, 'border_count': 32}
  Resultados salvos: optuna_results/semeion_catboost_best.json

[INFO] Tunando cnae-9...

Tuning CatBoost para: cnae-9
  Train: 680 amostras
  Features: 856
  Classes: 9
  Trials: 50


Best trial: 22. Best value: 0.995036: 100%|██████████| 50/50 [09:06<00:00, 10.93s/it]



  Melhor AUC (CV): 0.9950
  Tempo de tuning: 546.31s
  Melhores parâmetros: {'iterations': 452, 'learning_rate': 0.040359456141514524, 'depth': 10, 'l2_leaf_reg': 0.06035601781783563, 'random_strength': 3.2092601977965796, 'bagging_temperature': 0.838845022048059, 'border_count': 201}
  Resultados salvos: optuna_results/cnae-9_catboost_best.json

[INFO] Tunando vowel...

Tuning CatBoost para: vowel
  Train: 623 amostras
  Features: 12
  Classes: 11
  Trials: 50


...

Best trial: 3. Best value: 0.839521: 100%|██████████| 50/50 [00:13<00:00,  3.60it/s]



  Melhor AUC (CV): 0.8395
  Tempo de tuning: 13.90s
  Melhores parâmetros: {'iterations': 112, 'learning_rate': 0.027010527749605478, 'depth': 5, 'l2_leaf_reg': 0.23345864076016243, 'random_strength': 1.382623217936987, 'bagging_temperature': 0.19967378215835974, 'border_count': 147}
  Resultados salvos: optuna_results/diabetes_catboost_best.json

[INFO] Tunando ilpd...

Tuning CatBoost para: ilpd
  Train: 367 amostras
  Features: 10
  Classes: 2
  Trials: 50


Best trial: 21. Best value: 0.733935: 100%|██████████| 50/50 [00:23<00:00,  2.14it/s]


  Melhor AUC (CV): 0.7339
  Tempo de tuning: 23.35s
  Melhores parâmetros: {'iterations': 381, 'learning_rate': 0.015423467501110306, 'depth': 4, 'l2_leaf_reg': 0.179334650881117, 'random_strength': 9.776707423653912, 'bagging_temperature': 0.7088451718686236, 'border_count': 57}
  Resultados salvos: optuna_results/ilpd_catboost_best.json

RESUMO DO TUNING
Total de datasets: 30
Tempo total de tuning: 6940.40s
AUC médio (CV): 0.8979


,dataset,best_score,tune_time
25,analcatdata_authorship,1.000000,278.395276
4,banknote-authentication,1.000000,40.088494
24,tic-tac-toe,1.000000,18.706865
6,MiceProtein,0.999819,914.802143
19,car,0.999519,22.452347
10,vowel,0.998535,402.979586
8,semeion,0.997293,3552.299297
9,cnae-9,0.995036,546.307437
11,breast-w,0.994157,15.482778
13,wdbc,0.991388,74.933463


## Executar Treinamento

In [10]:
ensure_dirs()

# Lista completa dos 30 datasets
dataset_list = [
    # Datasets balanceados
    'credit-approval', 'dresses-sales', 'mfeat-morphological', 'vehicle',
    'banknote-authentication', 'analcatdata_dmft', 'MiceProtein', 'cylinder-bands',
    'semeion', 'cnae-9', 'vowel',
    # Datasets desbalanceados
    'breast-w', 'eucalyptus', 'wdbc', 'pc4', 'credit-g', 'cmc',
    'blood-transfusion-service-center', 'pc3', 'car', 'kc2',
    'steel-plates-fault', 'balance-scale', 'pc1', 'tic-tac-toe',
    'analcatdata_authorship', 'climate-model-simulation-crashes',
    'qsar-biodeg', 'diabetes', 'ilpd',
]

In [11]:
all_rows = []

for ds in dataset_list:
    try:
        # Verificar se os parâmetros já existem
        param_path = f"optuna_results/{ds}_catboost_best.json"
        
        if not os.path.exists(param_path):
            print(f"\n[INFO] Parâmetros não encontrados para {ds}. Executando tuning...")
            try:
                tune_catboost_optuna(ds, n_trials=50)
            except Exception as tune_err:
                print(f"[ERRO] Falha no tuning de {ds}: {tune_err}")
                continue
        
        # Treinar modelo com os melhores parâmetros
        rows = train_final_model(ds)
        all_rows.extend(rows)
        
    except FileNotFoundError as e:
        print(f"\n[ERRO] Dataset {ds} não encontrado: {e}")
        continue
    except Exception as e:
        print(f"\n[ERRO] Dataset {ds}: {e}")
        continue


Treinando CatBoost para: credit-approval
Melhores hiperparâmetros: {'iterations': 106, 'learning_rate': 0.06242587265091922, 'depth': 3, 'l2_leaf_reg': 0.2723574117325312, 'random_strength': 2.4057966497003065, 'bagging_temperature': 0.48431521159265845, 'border_count': 206}
  Train: 434 amostras
  Test: 207 amostras
  Features: 15
  Classes: 2
  Tempo de treino: 0.01s
  Modelo salvo: trained_models_catboost/credit-approval_catboost.cbm

  Resultados no TREINO:
    AUC OVO: 0.9700
    Accuracy: 0.9147
    G-Mean: 0.9155
    Cross-Entropy: 0.2493

  Resultados no TESTE:
    AUC OVO: 0.9280
    Accuracy: 0.8406
    G-Mean: 0.8420
    Cross-Entropy: 0.3475

Treinando CatBoost para: dresses-sales
Melhores hiperparâmetros: {'iterations': 173, 'learning_rate': 0.13614662198784633, 'depth': 7, 'l2_leaf_reg': 9.805962500133788, 'random_strength': 0.022281136225138456, 'bagging_temperature': 0.9275916119933421, 'border_count': 226}
  Train: 315 amostras
  Test: 150 amostras
  Features: 12
  Cl

## Salvar e Visualizar Resultados

In [12]:
if len(all_rows) > 0:
    df_results = pd.DataFrame(all_rows)
    output_path = "results/final_catboost_results.csv"
    df_results.to_csv(output_path, index=False)
    print(f"Resultados salvos em: {output_path}")
else:
    print("Nenhum resultado para salvar.")

Resultados salvos em: results/final_catboost_results.csv


In [13]:
# Exibir DataFrame completo
if len(all_rows) > 0:
    display(df_results)

,split,nome_modelo,dataset,tempo_tune,tempo_train,auc_ovo,mean_acc,g_mean,mean_cross_entropy,tempo_predict
0,train,CatBoost,credit-approval,14.668849,0.014622,0.970018,0.914747,0.915469,0.249339,0.000187
1,test,CatBoost,credit-approval,14.668849,0.014622,0.928022,0.840580,0.842020,0.347480,0.000108
2,train,CatBoost,dresses-sales,18.087480,0.027985,0.994103,0.939683,0.926759,0.328006,0.000168
3,test,CatBoost,dresses-sales,18.087480,0.027985,0.518154,0.526667,0.418510,0.743947,0.000110
4,train,CatBoost,mfeat-morphological,99.186992,0.203404,0.987884,0.838095,0.815146,0.405665,0.000584
5,test,CatBoost,mfeat-morphological,99.186992,0.203404,0.963341,0.710000,0.645019,0.641938,0.000773
6,train,CatBoost,vehicle,75.901436,0.041750,0.992620,0.941729,0.940630,0.334969,0.000295
7,test,CatBoost,vehicle,75.901436,0.041750,0.940860,0.795276,0.761709,0.476767,0.000184
8,train,CatBoost,banknote-authentication,40.088494,0.233166,1.000000,1.000000,1.000000,0.000086,0.000474
9,test,CatBoost,banknote-authentication,40.088494,0.233166,1.000000,1.000000,1.000000,0.000357,0.000277


### Resultados no Teste

In [14]:
if len(all_rows) > 0:
    test_results = df_results[df_results["split"] == "test"][
        ["dataset", "auc_ovo", "mean_acc", "g_mean", "mean_cross_entropy"]
    ].copy()
    test_results = test_results.sort_values("auc_ovo", ascending=False)
    display(test_results)

,dataset,auc_ovo,mean_acc,g_mean,mean_cross_entropy
9,banknote-authentication,1.000000,1.000000,1.000000,0.000357
49,tic-tac-toe,1.000000,1.000000,1.000000,0.011374
13,MiceProtein,0.999932,0.981481,0.975228,0.058066
21,vowel,0.999784,0.949495,0.954293,0.098170
39,car,0.999778,0.990366,0.989234,0.043932
51,analcatdata_authorship,0.999305,0.992095,0.981219,0.039678
17,semeion,0.998175,0.935146,0.935406,0.194377
27,wdbc,0.997942,0.976608,0.971372,0.054264
23,breast-w,0.997495,0.966667,0.959464,0.061169
19,cnae-9,0.995481,0.925926,0.924648,0.298727


### Estatísticas Resumidas

In [15]:
if len(all_rows) > 0:
    print("Estatísticas dos resultados no TESTE:")
    print("="*50)
    test_only = df_results[df_results["split"] == "test"]
    
    metrics = ["auc_ovo", "mean_acc", "g_mean", "mean_cross_entropy"]
    
    for m in metrics:
        print(f"\n{m}:")
        print(f"  Média: {test_only[m].mean():.4f}")
        print(f"  Desvio Padrão: {test_only[m].std():.4f}")
        print(f"  Mínimo: {test_only[m].min():.4f}")
        print(f"  Máximo: {test_only[m].max():.4f}")

Estatísticas dos resultados no TESTE:

auc_ovo:
  Média: 0.8927
  Desvio Padrão: 0.1273
  Mínimo: 0.5182
  Máximo: 1.0000

mean_acc:
  Média: 0.8262
  Desvio Padrão: 0.1724
  Mínimo: 0.2111
  Máximo: 1.0000

g_mean:
  Média: 0.6838
  Desvio Padrão: 0.2779
  Mínimo: 0.0000
  Máximo: 1.0000

mean_cross_entropy:
  Média: 0.3768
  Desvio Padrão: 0.3333
  Mínimo: 0.0004
  Máximo: 1.6044


### Tempos de Execução

In [16]:
if len(all_rows) > 0:
    print("Tempos de Execução:")
    print("="*50)
    
    # Pegar apenas uma linha por dataset (treino e teste têm os mesmos tempos)
    tempo_df = df_results[df_results["split"] == "test"][["dataset", "tempo_tune", "tempo_train", "tempo_predict"]]
    
    print(f"\nTempo total de tuning: {tempo_df['tempo_tune'].sum():.2f}s")
    print(f"Tempo total de treino: {tempo_df['tempo_train'].sum():.2f}s")
    print(f"Tempo médio de predição: {tempo_df['tempo_predict'].mean():.4f}s")
    
    display(tempo_df)

Tempos de Execução:

Tempo total de tuning: 6940.40s
Tempo total de treino: 44.27s
Tempo médio de predição: 0.0004s


,dataset,tempo_tune,tempo_train,tempo_predict
1,credit-approval,14.668849,0.014622,0.000108
3,dresses-sales,18.087480,0.027985,0.000110
5,mfeat-morphological,99.186992,0.203404,0.000773
7,vehicle,75.901436,0.041750,0.000184
9,banknote-authentication,40.088494,0.233166,0.000277
11,analcatdata_dmft,9.227260,0.027596,0.000144
13,MiceProtein,914.802143,2.781073,0.000506
15,cylinder-bands,40.821050,0.127134,0.000162
17,semeion,3552.299297,2.561040,0.000828
19,cnae-9,546.307437,32.078905,0.001301
